### Sources

- Pinecone URL - https://app.pinecone.io
- Getting Started - https://docs.pinecone.io/guides/get-started/overview
- Python SDK - https://docs.pinecone.io/reference/sdks/python/overview

### Setup

In [81]:
%pip install -q python-dotenv pinecone

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: C:\Users\dbenn\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [82]:
from dotenv import load_dotenv
from pinecone import Pinecone
import os

In [83]:
load_dotenv()
print("Pinecone API key loaded:", os.getenv("PINECONE_API_KEY") is not None)
pc_api = os.environ.get("PINECONE_API_KEY")

Pinecone API key loaded: True


In [84]:
# Initialize Pinecone client
pc = Pinecone(api_key=pc_api)

### Create Index

In [85]:
index_name = "code-demo"

if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model":"llama-text-embed-v2",
            "field_map":{"text": "chunk_text"}
        }
    )

### Upsert Data

In [86]:
records = [
    {"id": "1", "chunk_text": "Each player chooses one token to represent them on the board and begins at the GO space.", "category": "setup"},
    {"id": "2", "chunk_text": "Each player starts the game with $1,500 in Monopoly money, distributed as: two $500s, two $100s, two $50s, six $20s, five $10s, five $5s, and five $1s.", "category": "setup"},
    {"id": "3", "chunk_text": "Players take turns rolling two dice. The player with the highest total starts the game, and play proceeds clockwise.", "category": "gameplay"},
    {"id": "4", "chunk_text": "If you roll doubles, you move your token, take your action, and then roll again. If you roll doubles three times in a single turn, you must go directly to Jail.", "category": "gameplay"},
    {"id": "5", "chunk_text": "When you land on an unowned property, you may buy it for the price listed on the board. If you decline, the Banker must auction it immediately to the highest bidder.", "category": "property"},
    {"id": "6", "chunk_text": "If you land on a property owned by another player, you must pay them the rent amount shown on the Title Deed card. Rent increases significantly if the owner has a full color set or buildings.", "category": "property"},
    {"id": "7", "chunk_text": "You may build houses only after owning all properties in a color group. Houses must be built evenly across the set; you cannot place a second house on a property until all properties in that set have one.", "category": "buildings"},
    {"id": "8", "chunk_text": "Once you have four houses on each property in a color group, you may upgrade to a hotel by paying the cost and returning the four houses to the Bank.", "category": "buildings"},
    {"id": "9", "chunk_text": "You can be sent to Jail if you land on the 'Go to Jail' space, draw a 'Go to Jail' card, or roll doubles three times in one turn.", "category": "jail"},
    {"id": "10", "chunk_text": "To get out of Jail, you can pay a $50 fine, use a 'Get Out of Jail Free' card, or attempt to roll doubles on your turn.", "category": "jail"},
    {"id": "11", "chunk_text": "If you cannot pay your debts, you must mortgage unimproved properties. You receive half the printed value from the Bank and cannot collect rent on mortgaged properties.", "category": "bankruptcy"},
    {"id": "12", "chunk_text": "Bankruptcy occurs when you owe more than you can pay to either another player or the Bank. You must turn over all assets to the creditor and are eliminated from the game.", "category": "bankruptcy"},
    {"id": "13", "chunk_text": "Passing GO entitles you to collect a salary of $200 from the Bank.", "category": "gameplay"},
    {"id": "14", "chunk_text": "Railroads are owned properties where rent is determined by the number of railroads owned by that player: $25 for one, $50 for two, $100 for three, and $200 for four.", "category": "property"},
    {"id": "15", "chunk_text": "Utility rent is calculated by rolling the dice. If the owner has one utility, rent is 4x the dice roll; if they own both, it is 10x the dice roll.", "category": "property"},
    {"id": "16", "chunk_text": "Players may trade money, properties, and 'Get Out of Jail Free' cards with each other at any time during the game.", "category": "trading"},
    {"id": "17", "chunk_text": "You cannot sell buildings back to the Bank for full price; you receive only half of the purchase cost when selling houses or hotels.", "category": "buildings"},
    {"id": "18", "chunk_text": "Income Tax requires you to pay the Bank either $200 or 10% of your total net worth, whichever is lower.", "category": "spaces"},
    {"id": "19", "chunk_text": "Luxury Tax is a flat fee of $100 paid to the Bank when landing on the corresponding board space.", "category": "spaces"},
    {"id": "20", "chunk_text": "The game ends when all players except one have gone bankrupt; the last remaining player is the winner.", "category": "gameplay"}
]

In [87]:
index = pc.Index(index_name)
index.upsert_records(
  namespace="default",
  records=records
)

UpsertRecordsResponse(record_count=20, response_info=ResponseInfo(raw_headers={'date': 'Sat, 08 Aug 2026 15:04:02 GMT', 'content-length': '0', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '1', 'x-pinecone-api-version': '2025-10', 'x-envoy-upstream-service-time': '652', 'x-pinecone-response-duration-ms': '658', 'server': 'envoy'}))

In [88]:
stats = index.describe_index_stats()
for field in ("dimension", "total_vector_count"):
  print(f"{field}: {getattr(stats, field)}")

dimension: 1024
total_vector_count: 20


### Search

#### Semantic Search
Search a Pinecone index of dense vectors to find semantically similar records using text or vector queries, top_k results, and nearest neighbor lookup.

- https://docs.pinecone.io/guides/search/semantic-search?retry=2

In [89]:
# semantic search - Pinecone uses the embedding model integrated with the index to convert the text to a dense vector automatically.
results = index.search(
    namespace="default", 
    query={
        "inputs": {"text": "what happens when you pass go"}, 
        "top_k": 2
    },
    fields=["category", "chunk_text"]
)

In [90]:
for hit in results.result.hits:
  print(f"Score: {hit.score}")
  print(f"Category: {hit.fields['category']}")
  print(f"Text: {hit.fields['chunk_text']}")
  print("-" * 80)

Score: 0.37199196219444275
Category: gameplay
Text: Passing GO entitles you to collect a salary of $200 from the Bank.
--------------------------------------------------------------------------------
Score: 0.3518851697444916
Category: gameplay
Text: If you roll doubles, you move your token, take your action, and then roll again. If you roll doubles three times in a single turn, you must go directly to Jail.
--------------------------------------------------------------------------------


#### Rerank
Improve retrieval quality by reranking initial search results with a hosted or external model to surface the most relevant matches for RAG.
- https://docs.pinecone.io/guides/search/rerank-results

In [91]:
reranked_results = index.search(
    namespace="default",
    query={
            "inputs": {"text": "what happens when you pass go"}, 
            "top_k": 2
        },
    rerank={
        "model": "bge-reranker-v2-m3",
        "top_n": 2,
        "rank_fields": ["chunk_text"]
    },
    fields=["category", "chunk_text"]
)

In [92]:
for hit in reranked_results.result.hits:
  print(f"Score: {hit.score}")
  print(f"Category: {hit.fields['category']}")
  print(f"Text: {hit.fields['chunk_text']}")
  print("-" * 80)

Score: 0.9647889137268066
Category: gameplay
Text: Passing GO entitles you to collect a salary of $200 from the Bank.
--------------------------------------------------------------------------------
Score: 0.0061929901130497456
Category: gameplay
Text: If you roll doubles, you move your token, take your action, and then roll again. If you roll doubles three times in a single turn, you must go directly to Jail.
--------------------------------------------------------------------------------
